# Warehouse Robot with Q-Learning
Run all cells (Runtime → Run all). Each module cell writes a `.py` file, then the last cells train, evaluate and animate.

In [ ]:
!pip -q install numpy matplotlib pillow

## `environment.py`

In [ ]:
%%writefile environment.py
"""Warehouse environment (a simple grid world) for the Q-Learning robot."""
import numpy as np

# Actions
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3
ACTION_NAMES = ["UP", "DOWN", "LEFT", "RIGHT"]
MOVES = {UP: (-1, 0), DOWN: (1, 0), LEFT: (0, -1), RIGHT: (0, 1)}

# Rewards (each one has a purpose, explained in README / report)
R_STEP = -1            # small cost per move -> encourages short routes
R_OBSTACLE = -10       # hit an obstacle -> robot stays in place
R_WALL = -10           # tried to leave the grid -> robot stays in place
R_PICKUP = 20          # collected the package
R_DELIVER = 100        # delivered the package (episode ends)
R_EMPTY_STATION = -5   # reached delivery station without package

DEFAULT_OBSTACLES = [
    (1, 3), (2, 3), (3, 3), (4, 3), (4, 4), (4, 5),
    (6, 1), (6, 2), (6, 3), (6, 6), (7, 6), (8, 6),
    (2, 7), (3, 7), (3, 8), (5, 8), (8, 2), (8, 3),
]


class WarehouseEnv:
    """State = (row, col, has_package). Pickup/delivery happen automatically
    when the robot steps on the package / delivery cell."""

    def __init__(self, size=10, obstacles=None, start=(0, 0),
                 package=(7, 1), station=(9, 9), max_steps=200):
        self.size = size
        self.obstacles = set(DEFAULT_OBSTACLES if obstacles is None else obstacles)
        self.start, self.package, self.station = start, package, station
        self.max_steps = max_steps
        self.n_actions = 4
        self.n_states = size * size * 2
        for p in (start, package, station):
            assert p not in self.obstacles, f"{p} is on an obstacle"
        self.reset()

    # --- state <-> index helpers -------------------------------------
    def encode(self, row, col, has_pkg):
        return (row * self.size + col) * 2 + int(has_pkg)

    def reset(self):
        self.pos = self.start
        self.has_package = False
        self.delivered = False
        self.steps = 0
        return self.encode(*self.pos, self.has_package)

    def step(self, action):
        """Returns next_state, reward, done, info"""
        self.steps += 1
        dr, dc = MOVES[action]
        r, c = self.pos[0] + dr, self.pos[1] + dc
        reward = R_STEP
        info = {"obstacle_hit": False, "wall_hit": False}

        if not (0 <= r < self.size and 0 <= c < self.size):
            reward += R_WALL; info["wall_hit"] = True      # stay in place
        elif (r, c) in self.obstacles:
            reward += R_OBSTACLE; info["obstacle_hit"] = True  # stay in place
        else:
            self.pos = (r, c)
            if self.pos == self.package and not self.has_package and not self.delivered:
                self.has_package = True
                reward += R_PICKUP
            elif self.pos == self.station:
                if self.has_package:
                    self.has_package = False
                    self.delivered = True
                    reward += R_DELIVER
                else:
                    reward += R_EMPTY_STATION

        done = self.delivered or self.steps >= self.max_steps
        info["delivered"] = self.delivered
        return self.encode(*self.pos, self.has_package), reward, done, info

    def grid(self):
        """Integer grid for drawing: 0 empty, 1 obstacle"""
        g = np.zeros((self.size, self.size), dtype=int)
        for (r, c) in self.obstacles:
            g[r, c] = 1
        return g


## `q_learning.py`

In [ ]:
%%writefile q_learning.py
"""Q-Learning agent implemented from scratch with NumPy."""
import numpy as np


class QLearningAgent:
    def __init__(self, n_states, n_actions, alpha=0.1, gamma=0.95,
                 epsilon=1.0, epsilon_min=0.01, epsilon_decay=0.999, seed=0):
        self.q = np.zeros((n_states, n_actions))   # the Q-table
        self.alpha, self.gamma = alpha, gamma
        self.epsilon, self.epsilon_min, self.epsilon_decay = epsilon, epsilon_min, epsilon_decay
        self.n_actions = n_actions
        self.rng = np.random.default_rng(seed)

    def choose_action(self, state):
        """Epsilon-greedy: explore with probability epsilon, else exploit."""
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.n_actions))
        row = self.q[state]
        return int(self.rng.choice(np.flatnonzero(row == row.max())))  # random tie-break

    def update(self, s, a, r, s_next, done):
        """Q(s,a) = Q(s,a) + alpha * [r + gamma * max Q(s',a') - Q(s,a)]"""
        target = r if done else r + self.gamma * np.max(self.q[s_next])
        self.q[s, a] += self.alpha * (target - self.q[s, a])

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

    def best_action(self, state):
        return int(np.argmax(self.q[state]))

    def save(self, path):
        np.save(path, self.q)

    def load(self, path):
        self.q = np.load(path)


## `training.py`

In [ ]:
%%writefile training.py
"""Training loop for the Q-Learning warehouse robot."""
import numpy as np
from q_learning import QLearningAgent


def train(env, episodes=5000, alpha=0.1, gamma=0.95, epsilon_decay=0.999,
          seed=0, log_every=500, verbose=True):
    agent = QLearningAgent(env.n_states, env.n_actions, alpha, gamma,
                           epsilon_decay=epsilon_decay, seed=seed)
    history = {"reward": [], "steps": [], "success": [], "epsilon": []}
    for ep in range(1, episodes + 1):
        s = env.reset()
        total, done = 0, False
        while not done:
            a = agent.choose_action(s)
            s2, r, done, info = env.step(a)
            agent.update(s, a, r, s2, done)
            s, total = s2, total + r
        agent.decay_epsilon()
        history["reward"].append(total)
        history["steps"].append(env.steps)
        history["success"].append(int(env.delivered))
        history["epsilon"].append(agent.epsilon)
        if verbose and ep % log_every == 0:
            w = slice(-log_every, None)
            print(f"Episode {ep:5d} | avg reward {np.mean(history['reward'][w]):8.2f} | "
                  f"success {np.mean(history['success'][w])*100:5.1f}% | "
                  f"avg steps {np.mean(history['steps'][w]):6.1f} | eps {agent.epsilon:.3f}")
    return agent, history


## `visualization.py`

In [ ]:
%%writefile visualization.py
"""Warehouse drawing, learned-route plot and animation (works in Colab)."""
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.animation import FuncAnimation, PillowWriter

COL = {"empty": "#f4f1ea", "obstacle": "#34495e", "visited": "#aed6f1",
       "package": "#e67e22", "station": "#27ae60", "robot": "#e74c3c"}


def get_route(env, agent, max_steps=100):
    """Follow the greedy (best) action from the Q-table. Returns list of frames."""
    s = env.reset()
    frames = [(env.pos, env.has_package, env.delivered)]
    done = False
    while not done and env.steps < max_steps:
        s, r, done, info = env.step(agent.best_action(s))
        frames.append((env.pos, env.has_package, env.delivered))
    return frames


def draw_frame(ax, env, frames, k, title=""):
    ax.clear()
    n = env.size
    visited = {f[0] for f in frames[:k + 1]}
    for r in range(n):
        for c in range(n):
            color = COL["empty"]
            if (r, c) in env.obstacles: color = COL["obstacle"]
            elif (r, c) in visited: color = COL["visited"]
            ax.add_patch(Rectangle((c, n - 1 - r), 1, 1, fc=color, ec="#999", lw=.8))
    pr, pc = env.package; sr, sc = env.station
    pos, has_pkg, delivered = frames[k]
    if not has_pkg and not delivered:
        ax.add_patch(Rectangle((pc + .15, n - 1 - pr + .15), .7, .7, fc=COL["package"], ec="k"))
        ax.text(pc + .5, n - 1 - pr + .5, "P", ha="center", va="center", weight="bold")
    ax.add_patch(Rectangle((sc + .05, n - 1 - sr + .05), .9, .9, fc=COL["station"], ec="k", alpha=.9))
    ax.text(sc + .5, n - 1 - sr + .5, "D", ha="center", va="center", color="w", weight="bold")
    ax.add_patch(plt.Circle((pos[1] + .5, n - 1 - pos[0] + .5), .32, fc=COL["robot"], ec="k"))
    if has_pkg:
        ax.add_patch(Rectangle((pos[1] + .38, n - 1 - pos[0] + .38), .24, .24, fc=COL["package"], ec="k"))
    status = "DELIVERED!" if delivered else ("Carrying package" if has_pkg else "Going to package")
    ax.set_xlim(0, n); ax.set_ylim(0, n); ax.set_aspect("equal"); ax.axis("off")
    ax.set_title(f"{title}Step {k} - {status}")


def plot_route(env, agent, path):
    frames = get_route(env, agent)
    fig, ax = plt.subplots(figsize=(6, 6))
    draw_frame(ax, env, frames, len(frames) - 1, "Learned route | ")
    xs = [f[0][1] + .5 for f in frames]; ys = [env.size - 1 - f[0][0] + .5 for f in frames]
    ax.plot(xs, ys, "-", color="#c0392b", lw=2, alpha=.8)
    plt.savefig(path, dpi=150, bbox_inches="tight"); plt.close()
    return len(frames) - 1


def animate(env, agent, gif_path=None, interval=300):
    """Returns a FuncAnimation. In Colab: HTML(anim.to_jshtml())."""
    frames = get_route(env, agent)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    anim = FuncAnimation(fig, lambda k: draw_frame(ax, env, frames, k),
                         frames=len(frames), interval=interval, repeat=False)
    if gif_path:
        anim.save(gif_path, writer=PillowWriter(fps=3))
    plt.close(fig)
    return anim


def plot_warehouse(env, path=None):
    fig, ax = plt.subplots(figsize=(6, 6))
    draw_frame(ax, env, [(env.start, False, False)], 0, "Warehouse layout | ")
    if path: plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


## `evaluation.py`

In [ ]:
%%writefile evaluation.py
"""Compare the trained Q-Learning robot with a random-movement baseline."""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def run_policy(env, policy, episodes=500, seed=123):
    rng = np.random.default_rng(seed)
    res = {"reward": [], "steps": [], "success": [], "obstacle_hits": []}
    for _ in range(episodes):
        s, done, total, hits = env.reset(), False, 0, 0
        while not done:
            a = policy(s, rng)
            s, r, done, info = env.step(a)
            total += r
            hits += info["obstacle_hit"] or info["wall_hit"]
        res["reward"].append(total); res["steps"].append(env.steps)
        res["success"].append(int(env.delivered)); res["obstacle_hits"].append(hits)
    return res


def summarize(res):
    succ = np.array(res["success"]); steps = np.array(res["steps"])
    return {
        "Total successful deliveries": int(succ.sum()),
        "Success rate (%)": round(100 * succ.mean(), 1),
        "Avg steps (all episodes)": round(steps.mean(), 1),
        "Avg steps (successful only)": round(steps[succ == 1].mean(), 1) if succ.sum() else float("nan"),
        "Avg reward": round(float(np.mean(res["reward"])), 1),
        "Avg collisions per episode": round(float(np.mean(res["obstacle_hits"])), 1),
    }


def compare(env, agent, episodes=500):
    q_res = run_policy(env, lambda s, rng: agent.best_action(s), episodes)
    r_res = run_policy(env, lambda s, rng: int(rng.integers(4)), episodes)
    return summarize(q_res), summarize(r_res)


def print_table(q_sum, r_sum):
    print(f"{'Metric':35s}{'Q-Learning':>14s}{'Random':>14s}")
    for k in q_sum:
        print(f"{k:35s}{q_sum[k]:>14}{r_sum[k]:>14}")


def plot_training(history, path_dir):
    def smooth(x, w=50):
        return np.convolve(x, np.ones(w) / w, mode="valid")
    fig, ax = plt.subplots(2, 2, figsize=(12, 8))
    ax[0, 0].plot(smooth(history["reward"])); ax[0, 0].set_title("Reward per episode (moving avg)")
    ax[0, 1].plot(smooth(history["steps"])); ax[0, 1].set_title("Steps per episode (moving avg)")
    ax[1, 0].plot(smooth(history["success"]) * 100); ax[1, 0].set_title("Success rate % (moving avg)")
    ax[1, 1].plot(history["epsilon"]); ax[1, 1].set_title("Epsilon (exploration) decay")
    for a in ax.flat: a.set_xlabel("Episode"); a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(f"{path_dir}/training_progress.png", dpi=150); plt.close()


def plot_comparison(q_sum, r_sum, path_dir):
    keys = ["Success rate (%)", "Avg steps (all episodes)", "Avg reward", "Avg collisions per episode"]
    fig, ax = plt.subplots(1, 4, figsize=(15, 4))
    for a, k in zip(ax, keys):
        a.bar(["Q-Learning", "Random"], [q_sum[k], r_sum[k]], color=["#2e8b57", "#c0392b"])
        a.set_title(k, fontsize=10)
        for i, v in enumerate([q_sum[k], r_sum[k]]):
            a.text(i, v, f"{v}", ha="center", va="bottom")
    plt.tight_layout(); plt.savefig(f"{path_dir}/q_learning_vs_random.png", dpi=150); plt.close()


## Train, evaluate, visualise

In [ ]:
import os
os.makedirs('results/graphs', exist_ok=True); os.makedirs('models', exist_ok=True)
from environment import WarehouseEnv
from training import train
from evaluation import compare, print_table, plot_training, plot_comparison
from visualization import plot_route, animate, plot_warehouse
env = WarehouseEnv()
agent, history = train(env, episodes=5000)
agent.save('models/q_table.npy')
plot_training(history, 'results/graphs')
q_sum, r_sum = compare(env, agent, 500)
print_table(q_sum, r_sum)
plot_comparison(q_sum, r_sum, 'results/graphs')
print('Route length:', plot_route(env, agent, 'results/graphs/learned_route.png'))

In [ ]:
from IPython.display import HTML, Image, display
anim = animate(env, agent, 'results/graphs/robot_animation.gif')
display(HTML(anim.to_jshtml()))
for f in ['training_progress','q_learning_vs_random','learned_route']:
    display(Image(f'results/graphs/{f}.png'))